<a href="https://colab.research.google.com/github/antonDinkov/AI_integrationsForDev/blob/main/Exercise_Vector_Databases%2C_Embeddings_and_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q chromadb openai

In [3]:
from pathlib import Path
from pprint import pprint

def print_response(response):
    print(f"Reponse id: {response.id}")
    print(f"Status: {response.status}")
    print(f"Input tokens: {response.usage.input_tokens} ({response.usage.input_tokens_details.cached_tokens} cached) | Output tokens: {response.usage.output_tokens} ({response.usage.output_tokens_details.reasoning_tokens} reasoning)")
    pprint(response.output)

    print()
    print(f"{'=' * 20} [Text] {'=' * 20}")
    print(response.output_text)

In [4]:
from chromadb import PersistentClient

PATH_TO_CHROMA_DB = "/content/chromadb"
chroma_client = PersistentClient(path=PATH_TO_CHROMA_DB)
chroma_collection = chroma_client.get_or_create_collection(name="books")

In [5]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get('OPEN_AI_API_KEY')
openai_client = OpenAI(api_key=api_key)

In [6]:
file_paths = [Path("/content/Metamorphosis.txt"), Path("/content/Moby Dick.txt"), Path("/content/Alice's Adventures in Wonderland.txt")]

# Index the book

In [7]:
def index_file(path_to_file):
    with path_to_file.open() as file:
        lines = file.readlines()

    lines = [{ "text": line.strip(), "row": i + 1 } for i, line in enumerate(lines)]
    lines = [line for line in lines if line["text"] != '']

    line_idx = 0
    chunk_length = 6
    chunk_overlap = 2
    chunks = []

    while line_idx < len(lines):
        take = min(chunk_length, len(lines) - line_idx)
        start = line_idx

        chunks.append({
            "text": ' '.join(lines[start + i]["text"] for i in range(take)),
            "row_range": { "from": lines[start]["row"], "to": lines[start + take - 1]["row"] }
        })

        line_idx = line_idx + chunk_length - chunk_overlap


    chroma_collection.add(
        ids=[f"{path_to_file.stem}_{chunk["row_range"]["from"]}_{chunk["row_range"]["to"]}" for chunk in chunks],
        metadatas=[{ "book_name": path_to_file.stem, "ref_start": chunk["row_range"]["from"], "ref_end": chunk["row_range"]["to"] } for chunk in chunks],
        documents=[chunk["text"] for chunk in chunks]
    )

In [8]:
from rich.console import Console
from rich.table import Table

def print_query_results(result):
    console = Console()

    queries_count = len(result["ids"])
    for i in range(queries_count):
        table = Table(show_lines=True, expand=True)
        table.add_column("#")
        table.add_column("Text")

        results_count = len(result["ids"][i])
        for j in range(results_count):
            table.add_row(result["ids"][i][j], result["documents"][i][j])

        console.print(table)

In [9]:
for i in range(len(file_paths)):
    index_file(file_paths[i])

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 25.5MiB/s]


In [10]:
queries = ["The nature is beautiful", "In my room is dark", "It is amazing how a person can change"]

In [11]:
semantic_search_results = chroma_collection.query(
    query_texts=queries
)

In [12]:
print_query_results(semantic_search_results)

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #                     ┃ Text                                                                                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Moby Dick_15173_15179 │ is very sweet and rich; it has been tasted by man; it might do well with strawberries.  │
│                       │ When overflowing with mutual esteem, the whales salute _more hominum_. And thus, though │
│                       │ surrounded by circle upon circle of consternations and affrights, did these inscrutable │
│                       │ creatures at the centre freely and fearlessly indulge in all peaceful concernments;     │
│                       │ yea, serenely revelled                                                                  │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_3482_3487   │ these things unite in a man of greatly superior natural force, with a globular brain    │
│                       │ and a ponderous heart; who has also by the stillness and seclusion of many long         │
│                       │ night-watches in the remotest waters, and beneath constellations never seen here at the │
│                       │ north, been led to think untraditionally and independently; receiving all nature’s      │
│                       │ sweet or savage impressions fresh from her own virgin voluntary and confiding           │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_903_908     │ quietest, most enchanting bit of romantic landscape in all the valley of the Saco. What │
│                       │ is the chief element he employs? There stand his trees, each with a hollow trunk, as if │
│                       │ a hermit and a crucifix were within; and here sleeps his meadow, and there sleep his    │
│                       │ cattle; and up from yonder cottage goes a sleepy smoke. Deep into distant woodlands     │
│                       │ winds a mazy way, reaching to overlapping spurs of mountains bathed in                  │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_17389_17394 │ the trees stood high and haughty, feeling their living sap; the industrious earth       │
│                       │ beneath was as a weaver’s loom, with a gorgeous carpet on it, whereof the ground-vine   │
│                       │ tendrils formed the warp and woof, and the living flowers the figures. All the trees,   │
│                       │ with all their laden branches; all the shrubs, and ferns, and grasses; the              │
│                       │ message-carrying air; all these unceasingly were active. Through the                    │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_18573_18578 │ the world, the Indian ocean and Atlantic being but its arms. The same waves wash the    │
│                       │ moles of the new-built Californian towns, but yesterday planted by the recentest race   │
│                       │ of men, and lave the faded but still gorgeous skirts of Asiatic lands, older than       │
│                       │ Abraham; while all between float milky-ways of coral isles, and low-lying, endless,     │
│                       │ unknown Archipelagoes, and impenetrable Japans. Thus this mysterious, divine            │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_16256_16261 │ a most refreshing, convivial, beautiful object to behold. As its name imports, it is of │
│                       │ an exceedingly rich, mottled t

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #                     ┃ Text                                                                                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Moby Dick_2748_2753   │ keeping my eyes shut, in order the more to concentrate the snugness of being in bed.    │
│                       │ Because no man can ever feel his own identity aright except his eyes be closed; as if   │
│                       │ darkness were indeed the proper element of our essences, though light be more congenial │
│                       │ to our clayey part. Upon opening my eyes then, and coming out of my own pleasant and    │
│                       │ self-created darkness into the imposed and coarse outer gloom of the                    │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_2752_2757   │ part. Upon opening my eyes then, and coming out of my own pleasant and self-created     │
│                       │ darkness into the imposed and coarse outer gloom of the unilluminated                   │
│                       │ twelve-o’clock-at-night, I experienced a disagreeable revulsion. Nor did I at all       │
│                       │ object to the hint from Queequeg that perhaps it were best to strike a light, seeing    │
│                       │ that we were so wide awake; and besides he felt a strong desire to have a few quiet     │
│                       │ puffs                                                                                   │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_2638_2643   │ As I sat there in that now lonely room; the fire burning low, in that mild stage when,  │
│                       │ after its first intensity has warmed the air, it then only glows to be looked at; the   │
│                       │ evening shades and phantoms gathering round the casements, and peering in upon us       │
│                       │ silent, solitary twain; the storm booming without in solemn swells; I began to be       │
│                       │ sensible of strange feelings. I felt a melting in me. No more my splintered heart       │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_16540_16546 │ seeks the food of light, so he lives in light. He makes his berth an Aladdin’s lamp,    │
│                       │ and lays him down in it; so that in the pitchiest night the ship’s black hull still     │
│                       │ houses an illumination. See with what entire freedom the whaleman takes his handful of  │
│                       │ lamps—often but old bottles and vials, though—to the copper cooler at the try-works,    │
│                       │ and replenishes them there, as mugs of ale at a vat. He                                 │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_14618_14624 │ my head. The invariable moisture of my hair, while plunged in deep thought, after six   │
│                       │ cups of hot tea in my thin shingled attic, of an August noon; this seems an additional  │
│                       │ argument for the above supposition. And how nobly it raises our conceit of the mighty,  │
│                       │ misty monster, to behold him solemnly sailing through a calm tropical sea; his vast,    │
│                       │ mild                                                                                    │
├───────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────┤
│ Moby Dick_19281_19288 │ all is blackness of doom; but 

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #                                        ┃ Text                                                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Metamorphosis_382_388                    │ reason for the change, though we knew it tied in somehow with our    │
│                                          │ stay on TR768-L-14, and probably with the things that had bitten us. │
│                                          │ The cause was of secondary importance; the marvel of the reality was │
│                                          │ what intrigued us. We looked forward with poorly restrained          │
│                                          │ excitement to displaying our new mental and physical dexterity. *    │
│                                          │ *       *       *       *                                            │
├──────────────────────────────────────────┼──────────────────────────────────────────────────────────────────────┤
│ Metamorphosis_311_317                    │ Roesler-Zealley had noted the brief play of understanding on my face │
│                                          │ and he nodded. "I had to be certain, Max," he said. "You've changed  │
│                                          │ too, you know." Which was true. The mites in our veins had altered   │
│                                          │ us both considerably through the years. We had developed some small  │
│                                          │ empathy with them and they often performed as we wished. It was not  │
│                                          │ that they could read                                                 │
├──────────────────────────────────────────┼──────────────────────────────────────────────────────────────────────┤
│ Alice's Adventures in Wonderland_696_704 │ could. "No," said the Caterpillar. It unfolded its arms, took the    │
│                                          │ hookah out of its mouth again, and said, "So you think you're        │
│                                          │ changed, do you?" "I'm afraid, I am, sir," said Alice. "I can't      │
│                                          │ remember things as I used--and I don't keep the same size for ten    │
│                                          │ minutes together!"                                                   │
├──────────────────────────────────────────┼──────────────────────────────────────────────────────────────────────┤
│ Alice's Adventures in Wonderland_709_716 │ doesn't like changing so often, you know. I should like to be a      │
│                                          │ _little_ larger, sir, if you wouldn't mind," said Alice. "Three      │
│                                          │ inches is such a wretched height to be." "It is a very good height   │
│                                          │ indeed!" said the Caterpillar angrily, rearing itself upright as it  │
│                                          │ spoke (it was exactly three inches high). In a minute or two, the    │
│                                          │ Caterpillar got down off the mushroom and                            │
├──────────────────────────────────────────┼──────────────────────────────────────────────────────────────────────┤
│ Moby Dick_850_855                        │ regulating the circulation. Whenever I find myself growing grim      │
│                                          │ about the mouth; whenever it is a damp, drizzly November in my soul; │
│                                          │ whenever I find myself involuntarily pausing before coffin           │
│                                          │ warehouses, and bringing up the rear of every funeral I meet; and    │
│                                          │ especially 

# RAG

In [13]:
from pydantic import BaseModel, Field

class QuoteLookupRequest(BaseModel):
    """
    Searches the quotes catalog using semantic similarity to find phrases
    matching the user's intent. Use this tool when a user asks about
    the content of a book, a quote reference, or recommendations.
    """
    query: str = Field(description="A natural language search query describing the desired quote (e.g. 'leadership as the most important characterstic of the modern man', 'love will save the world', 'motivation is all you need')")


In [14]:
quote_lookup_request_schema = QuoteLookupRequest.model_json_schema()

tools = [
    {
        "type": "function",
        "name": "lookup_quote",
        "description": quote_lookup_request_schema["description"],
        "parameters": {
          "type": "object",
          "properties": quote_lookup_request_schema["properties"],
          "required": [*quote_lookup_request_schema["properties"].keys()],
          "additionalProperties": False
        },
        "strict": True
    }
]

In [15]:
def lookup_quote(req: QuoteLookupRequest):
    query_result = chroma_collection.query(query_texts=[req.query], include=["documents"])

    result_documents_count = len(query_result["ids"][0])

    # TODO: split id in book name and row
    return [{ "id": query_result["ids"][0][i], "text": query_result["documents"][0][i] } for i in range(result_documents_count)]


tool_handlers = {
    "lookup_quote": lambda args: lookup_quote(QuoteLookupRequest(**args))
}

In [16]:
import json

def interact_with_ai(input, max_iterations_count = 10):
    full_conversation = [*input]

    for i in range(max_iterations_count):
        print(f"Starting iteration #{i + 1}")

        current_response = openai_client.responses.create(
            model="gpt-5-nano",
            input=full_conversation,
            tools=tools
        )

        # TODO: sanitize
        full_conversation.extend(current_response.output)

        tool_calls = [item for item in current_response.output if item.type == "function_call"]
        if len(tool_calls) == 0:
            return full_conversation

        print(f"Found {len(tool_calls)} tool calls.")

        for tc in tool_calls:
            print(tc.name, tc.arguments)
            handler = tool_handlers[tc.name]
            result = handler(json.loads(tc.arguments))

            full_conversation.append({
                "type": "function_call_output",
                "call_id": tc.call_id,
                "output": json.dumps(result)
            })

            print(full_conversation[-1])


    raise Exception(f"The AI interaction couldn't finish in {max_iterations_count} iterations.")

In [17]:
system_prompt = "You are an expert in book recommendations and quote finding. The user will ask you to lookup quotes on a given topic. Always use the \"lookup_query\" tool to find interesting references. The final response should be based solely on the \"lookup_query\" tool call output you get. Do not invent or refer to any quotes that are not part of the \"lookup_query\" output."
user_prompts = [
    "I need to find a quote about the transformational power of love and beauty.",
    "Find a picturesque quote about childhood, dreaming and living effortlessly.",
    "Find quotes about the meaning of life and combine them with other quotes about simple existential obstacles to make the reader feel more motivated.",
    "Find the most dark soul breaking quotes in existence."
]

In [18]:
for user_prompt in user_prompts:
    conversation = interact_with_ai(
        input=[
            { "role": "developer", "content": system_prompt },
            { "role": "user", "content": user_prompt }
        ]
    )

    print("\n".join(item.text for item in conversation[-1].content))

Starting iteration #1
Found 1 tool calls.
lookup_quote {"query":"transformational power of love and beauty"}
{'type': 'function_call_output', 'call_id': 'call_IDd74jXptmGDMimLPDZ8Bldi', 'output': '[{"id": "Moby Dick_14678_14683", "text": "appalling beauty from it. Real strength never impairs beauty or harmony, but it often bestows it; and in everything imposingly beautiful, strength has much to do with the magic. Take away the tied tendons that all over seem bursting from the marble in the carved Hercules, and its charm would be gone. As devout Eckerman lifted the linen sheet from the naked corpse of Goethe, he was overwhelmed with"}, {"id": "Moby Dick_7955_7960", "text": "and the gilded velvets of butterflies, and the butterfly cheeks of young girls; all these are but subtile deceits, not actually inherent in substances, but only laid on from without; so that all deified Nature absolutely paints like the harlot, whose allurements cover nothing but the charnel-house within; and when we